In [ ]:
# %pip install -U -q langchain-mcp-adapters

In [20]:
from dotenv import load_dotenv

load_dotenv()

True

In [21]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient

github_pat = os.getenv("GITHUB_PAT")

mcp_client = MultiServerMCPClient({
     "github": {
      "transport": "streamable_http",
      "url": "https://api.githubcopilot.com/mcp/",
      "headers": {
        "Authorization": f"Bearer {github_pat}"
      },
     }
})



In [22]:
tool_list = await mcp_client.get_tools()

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

In [ ]:
from langgraph.prebuilt import create_agent

agent = create_agent(
    model=llm,
    tools=tool_list,
    system_prompt="Use the tools provided to you to answer the user's question"
)

In [ ]:
async def process_stream(stream_generator):
    results = []
    try:
        async for chunk in stream_generator:

            key = list(chunk.keys())[0]
            
            if key == 'agent':
                # Agent 메시지의 내용을 가져옴. 메세지가 비어있는 경우 어떤 도구를 어떻게 호출할지 정보를 가져옴
                content = chunk['agent']['messages'][0].content if chunk['agent']['messages'][0].content != '' else chunk['agent']['messages'][0].additional_kwargs
                print(f"'agent': '{content}'")
            
            elif key == 'tools':
                # 도구 메시지의 내용을 가져옴
                for tool_msg in chunk['tools']['messages']:
                    print(f"'tools': '{tool_msg.content}'")
            
            results.append(chunk)
        return results
    except Exception as e:
        print(f"Error processing stream: {e}")
        return results

In [ ]:
from langchain_core.messages import HumanMessage

query = """깃헙의 Pull Request를 확인하고 코드 리뷰를 작성해주세요. 
코드리뷰를 PR에 comment로 남겨주세요 

PR URL:https://https://github.com/baba-ti/Ai-agent-basic/pull/4

"""

stream_generator = agent.astream({'messages': [HumanMessage(content=query)]})


all_chunks = await process_stream(stream_generator)


if all_chunks:
    final_result = all_chunks[-1]
    print(final_result)